# 11. Biomechanical Proxy Test

Pipeline step ⑨. Verifies `extract_rep_biomech()` on normalized squat data.

The biomechanical proxy model estimates:
- **CoM (Center of Mass)** range and path length — from statistical anthropometric segment weights
- **Moment arm proxy** — knee and hip sagittal-plane perpendicular distances

All outputs are relative (`torso_length_ratio`). Absolute force units (N·m, kg)
are never produced; if they appear it is a bug.

Pipeline position: Feature Extraction → **Biomech Proxy** → Biomarker Derivation

This notebook assumes that the following notebooks are already passing:
- 00_environment_check through 10_feature_extraction_test

Reference: `docs/pipeline/09_biomechanical_proxy.md`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.biomech import BiomechRecord, extract_rep_biomech
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig, BiomechConfig, ExerciseDefinitionConfig, FeaturesConfig,
    MotionAttributionConfig, NormalizationConfig, PhaseSegmentationConfig,
    PipelineConfig, ValidationConfig, run_pipeline,
)
from movement.segmentation import segment_phases
from movement.validation import run_basic_validation

print('imports OK')

## Data Setup

Runs pipeline ①–⑥ to produce normalized + phase-labeled data.

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)
run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
ann_df       = load_annotation_csv(ann_path)
df_ann, _    = apply_annotation(df_raw, ann_df)
exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
df_norm, _   = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)
df_seg, _    = segment_phases(df_norm, exercise_def, fps_default=30.0)

print(f'data ready: {df_seg.shape[0]} frames, {df_seg["rep_id"].dropna().nunique()} reps')

## Direct extract_rep_biomech() Test

In [ ]:
records = extract_rep_biomech(df_seg, exercise_def, use_visibility_weight=True)

print(f'BiomechRecord count: {len(records)}')
for r in records:
    print(f'  rep={r.rep_id}  {r.metric_id:40s}  {r.value:.4f} {r.unit}')

## Check 1: BiomechRecord Fields

In [ ]:
for r in records:
    assert isinstance(r, BiomechRecord)
    assert r.metric_id,               f'metric_id empty'
    assert r.rep_id is not None,      f'rep_id None'
    assert r.value is not None,       f'value None for {r.metric_id}'
    assert r.unit,                    f'unit empty'
    assert len(r.source_fields) > 0,  f'source_fields empty for {r.metric_id}'
print(f'PASS: all {len(records)} BiomechRecord fields valid')

## Check 2: No Absolute Units

In [ ]:
import re
forbidden = re.compile(r'\b(N|Nm|N\.m|kg|meter|newton)\b', re.IGNORECASE)
for r in records:
    assert not forbidden.search(r.unit), f'absolute unit in {r.metric_id}: {r.unit}'
print('PASS: no absolute units (all relative torso_length_ratio or dimensionless)')
print(f'units found: {set(r.unit for r in records)}')

## Check 3: CoM and Moment Arm Records iresent

In [ ]:
metric_ids = [r.metric_id for r in records]
has_com    = any('com' in m for m in metric_ids)
has_ma     = any('moment_arm' in m for m in metric_ids)
print(f'CoM metrics   : {[m for m in metric_ids if "com" in m]}')
print(f'Moment arm    : {[m for m in metric_ids if "moment_arm" in m]}')
assert has_com, 'no CoM metrics found'
assert has_ma,  'no moment arm metrics found'
print('PASS: both CoM and moment arm metrics present')

## Check 4: Visibility Weighting Applied

In [ ]:
for r in records:
    assert r.visibility_weight_applied is not None, f'visibility_weight_applied missing for {r.metric_id}'
    assert r.n_frames_used is not None,             f'n_frames_used missing'
    assert r.n_frames_used > 0,                     f'n_frames_used == 0 for {r.metric_id}'
print('PASS: visibility weighting fields present and valid')
for r in records[:3]:
    print(f'  {r.metric_id}: n_frames_used={r.n_frames_used}  '
          f'n_excluded={r.n_frames_excluded_low_visibility}')

## Check 5: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True)
cfg.motion_attribution  = MotionAttributionConfig(enabled=True)
cfg.features            = FeaturesConfig(enabled=True)
cfg.biomech             = BiomechConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS)

assert 'biomech' in pipe_report
n_bio = len(pipe_report['biomech'])
print(f'PASS: pipeline ⑨ biomech report: {n_bio} records')
print(f'steps executed: {list(pipe_report.keys())}')

## Interpretation

| Metric type | Expected keys | Unit |
|---|---|---|
| CoM displacement | `com.range_x`, `com.range_z`, `com.path_length` | `torso_length_ratio` |
| Knee moment arm | `moment_arm.knee_sagittal.*` | `torso_length_ratio` |
| Hip moment arm | `moment_arm.hip_sagittal.*` | `torso_length_ratio` |

**Biomechanical interpretation**: CoM path length reflects total movement
volume of the body's mass center. A longer path relative to torso length may
indicate less direct movement or compensatory trunk displacement.

**Scope**: These are relative load-distribution *tendencies*, not absolute torques.
They cannot be used to infer joint contact forces or muscle activation levels.